# Tutorial 15: Theory — Symbolic Dynamics Capstone

**Tier 4 — System Dynamics**

This final tutorial lifts HLLSet evolution into the language of
**symbolic dynamics** and **ergodic theory**.

The key insight: the (D, R, N) transformation on HLLSet bit vectors
is a **generalized shift** on the binary sequence space $\{0,1\}^N$.
This connects HLLSet algebra to:

- **Bernoulli maps** — the canonical chaotic dynamical system
- **De Bruijn walks** — trajectories through state space
- **Lyapunov exponents** — quantifying sensitivity to perturbation
- **Poincaré recurrence** — when does the system revisit a state?

## What You'll Learn

1. **BitVectorState** — View HLLSets as points in $\{0,1\}^N$
2. **ShiftTransition** — D/R/N as bit-level clear/set masks
3. **BernoulliAnalyzer** — Entropy, mixing rate, Lyapunov, recurrence
4. **UnifiedSystemController** — Bridging HLLSet algebra with symbolic dynamics
5. **Walk Complexity** — Characterizing trajectory structure

## Prerequisites

| Tutorial | Concept |
|----------|---------|
| [09_hllset_debruijn.ipynb](09_hllset_debruijn.ipynb) | D/R/N triples, De Bruijn graph |
| [11_hllset_debruijn.ipynb](11_hllset_debruijn.ipynb) | Evolution phases |
| [13_observe.ipynb](13_observe.ipynb) | Noether conservation |
| [14_act.ipynb](14_act.ipynb) | Planning and steering |

## The Bernoulli Connection

```
   Classical Bernoulli Map          HLLSet Shift Map
   ─────────────────────           ──────────────────
   x ∈ [0, 1)                     v ∈ {0,1}^N
   x ↦ 2x mod 1                   v ↦ (v AND NOT D) OR N
   Binary digits shift left        Bit positions cleared/set
   De Bruijn graph = orbits        D/R/N = generalized shift
   Entropy = log(2)                Entropy ≈ H(density)
```

The registers of an HLLSet form a bit vector of size $2^P \times 32$.
Each (D, R, N) step clears some bits (D) and sets others (N),
exactly like a shift in the symbolic dynamics sense.

In [1]:
# Setup
import sys
sys.path.insert(0, '..')

from core.hllset_dynamics import (
    BitVectorState,
    ShiftTransition,
    BernoulliAnalyzer,
    UnifiedSystemController,
    SteeringMode,
)
from core.hllset import HLLSet
from core.bitvector_ring import BitVector
from core.bss import bss
from core.hllset_debruijn import decompose_transition, classify_transition

P_BITS = 10
N_BITS = (1 << P_BITS) * 32  # Total bits in HLLSet register space
print(f"State space: {{0,1}}^{N_BITS} ({N_BITS} bits = {N_BITS//8} bytes)")

State space: {0,1}^32768 (32768 bits = 4096 bytes)


## 1. BitVectorState — HLLSets as Binary Strings

A `BitVectorState` wraps the raw bit vector of an HLLSet,
exposing properties from symbolic dynamics:

In [2]:
# Create two HLLSets and convert to BitVectorState
h1 = HLLSet.from_batch(["alpha", "beta", "gamma", "delta"], p_bits=P_BITS)
h2 = HLLSet.from_batch(["alpha", "beta", "epsilon", "zeta"], p_bits=P_BITS)

s1 = BitVectorState.from_hllset(h1)
s2 = BitVectorState.from_hllset(h2)

print(f"State s1:")
print(f"  popcount (Hamming weight): {s1.popcount}")
print(f"  density (fraction of 1s):  {s1.density:.6f}")
print(f"  n_bits:                    {s1.n_bits}")

print(f"\nState s2:")
print(f"  popcount: {s2.popcount}")
print(f"  density:  {s2.density:.6f}")

# Distances
print(f"\nHamming distance:     {s1.hamming_distance(s2)}")
print(f"Normalized distance:  {s1.normalized_distance(s2):.6f}")

State s1:
  popcount (Hamming weight): 4
  density (fraction of 1s):  0.000122
  n_bits:                    32768

State s2:
  popcount: 4
  density:  0.000122

Hamming distance:     4
Normalized distance:  0.000122


## 2. Ring Operations on Bit Vectors

`BitVectorState` supports the Boolean ring operations:
- XOR = ring addition (symmetric difference)
- AND = ring multiplication (intersection)
- OR  = union

In [3]:
# Ring operations
xor_state = s1.xor(s2)     # Symmetric difference
and_state = s1.and_(s2)    # Intersection
or_state  = s1.or_(s2)     # Union

print(f"s1 XOR s2  (symmetric diff): popcount = {xor_state.popcount}, density = {xor_state.density:.6f}")
print(f"s1 AND s2  (intersection):   popcount = {and_state.popcount}, density = {and_state.density:.6f}")
print(f"s1 OR  s2  (union):          popcount = {or_state.popcount},  density = {or_state.density:.6f}")

# Verify: XOR popcount = Hamming distance
print(f"\nVerify: XOR popcount ({xor_state.popcount}) = Hamming distance ({s1.hamming_distance(s2)})")

# Verify: |A OR B| = |A| + |B| - |A AND B| (inclusion-exclusion on popcount)
ie_check = s1.popcount + s2.popcount - and_state.popcount
print(f"Inclusion-exclusion: {s1.popcount} + {s2.popcount} - {and_state.popcount} = {ie_check}")
print(f"Actual OR popcount:  {or_state.popcount}")

s1 XOR s2  (symmetric diff): popcount = 4, density = 0.000122
s1 AND s2  (intersection):   popcount = 2, density = 0.000061
s1 OR  s2  (union):          popcount = 6,  density = 0.000183

Verify: XOR popcount (4) = Hamming distance (4)
Inclusion-exclusion: 4 + 4 - 2 = 6
Actual OR popcount:  6


## 3. ShiftTransition — D/R/N at the Bit Level

A `ShiftTransition` is the bit-level representation of (D, R, N):

$$v' = (v \wedge \lnot \text{clear\_mask}) \vee \text{set\_mask}$$

where `clear_mask` = D (bits to delete) and `set_mask` = N (bits to add):

In [4]:
# Compute shift transition from s1 → s2
shift = ShiftTransition.from_states(s1, s2)

print(f"ShiftTransition s1 → s2:")
print(f"  clear_mask popcount: {shift.clear_mask.popcount()} (bits to delete)")
print(f"  set_mask popcount:   {shift.set_mask.popcount()} (bits to add)")
print(f"  Hamming cost:        {shift.hamming_cost} (total bits changed)")
print(f"  Is identity:         {shift.is_identity}")

# Apply the shift
s2_reconstructed = shift.apply(s1)
print(f"\nVerification:")
print(f"  Original s2 popcount:      {s2.popcount}")
print(f"  Reconstructed s2 popcount: {s2_reconstructed.popcount}")
print(f"  Distance after apply:      {s2.hamming_distance(s2_reconstructed)}")

ShiftTransition s1 → s2:
  clear_mask popcount: 2 (bits to delete)
  set_mask popcount:   2 (bits to add)
  Hamming cost:        4 (total bits changed)
  Is identity:         False

Verification:
  Original s2 popcount:      4
  Reconstructed s2 popcount: 4
  Distance after apply:      0


In [5]:
# Convert a DRNTriple to a ShiftTransition
drn = decompose_transition(h1, h2)
shift_from_drn = ShiftTransition.from_drn(drn, p_bits=P_BITS)

print(f"From DRNTriple:")
print(f"  D cardinality ≈ {drn.deleted_card:.1f}  →  clear_mask popcount = {shift_from_drn.clear_mask.popcount()}")
print(f"  N cardinality ≈ {drn.novel_card:.1f}  →  set_mask popcount   = {shift_from_drn.set_mask.popcount()}")
print(f"  Hamming cost = {shift_from_drn.hamming_cost}")

From DRNTriple:
  D cardinality ≈ 3.0  →  clear_mask popcount = 2
  N cardinality ≈ 4.0  →  set_mask popcount   = 2
  Hamming cost = 4


## 4. BernoulliAnalyzer — Ergodic Analysis Engine

The `BernoulliAnalyzer` treats a sequence of HLLSet states as a
trajectory in the symbolic dynamical system and computes:

| Metric | Meaning |
|--------|---------|
| `density_series()` | Fraction of 1-bits at each step |
| `entropy_estimate()` | Binary entropy $H(p)$ from average density |
| `mixing_rate()` | How many bits change per transition (normalized) |
| `recurrence_time()` | Steps until system revisits initial neighborhood |
| `lyapunov_estimate()` | Sensitivity to perturbation (expansion rate) |
| `walk_complexity()` | Full structural statistics |

In [6]:
# Build a trajectory: evolving document corpus
analyzer = BernoulliAnalyzer(n_bits=N_BITS)

# 15-step evolution with varied dynamics
import random
random.seed(42)

vocab = [f"word_{i}" for i in range(200)]
current_tokens = set(random.sample(vocab, 20))

for step in range(15):
    state = HLLSet.from_batch(list(current_tokens), p_bits=P_BITS)
    analyzer.observe_hllset(state)
    
    # Evolve: add 3-6 tokens, remove 1-4
    n_add = random.randint(3, 6)
    n_del = random.randint(1, 4)
    to_add = set(random.sample(vocab, n_add))
    to_del = set(random.sample(list(current_tokens), min(n_del, len(current_tokens))))
    current_tokens = (current_tokens - to_del) | to_add

print(f"Trajectory: {len(analyzer.trajectory)} states, {len(analyzer.transitions)} transitions")

Trajectory: 15 states, 14 transitions


## 5. Density Series — The System's Pulse

In [7]:
densities = analyzer.density_series()

print(f"Bit density over time (fraction of 1-bits):")
print(f"{'Step':>4}  {'Density':>10}  Visualization")
print("-" * 50)
for i, d in enumerate(densities):
    bar_len = int(d * 5000)  # Scale for visibility
    bar = "█" * min(bar_len, 40)
    print(f"{i:4d}  {d:10.6f}  {bar}")

import statistics
print(f"\nDensity statistics:")
print(f"  Mean:   {statistics.mean(densities):.6f}")
print(f"  Stdev:  {statistics.stdev(densities):.6f}")
print(f"  Min:    {min(densities):.6f}")
print(f"  Max:    {max(densities):.6f}")

Bit density over time (fraction of 1-bits):
Step     Density  Visualization
--------------------------------------------------
   0    0.000610  ███
   1    0.000641  ███
   2    0.000641  ███
   3    0.000671  ███
   4    0.000732  ███
   5    0.000732  ███
   6    0.000793  ███
   7    0.000793  ███
   8    0.000885  ████
   9    0.000854  ████
  10    0.000946  ████
  11    0.001007  █████
  12    0.001068  █████
  13    0.001129  █████
  14    0.001190  █████

Density statistics:
  Mean:   0.000846
  Stdev:  0.000187
  Min:    0.000610
  Max:    0.001190


## 6. Entropy — Information Content of the Trajectory

The binary entropy $H(p) = -p \log_2 p - (1-p) \log_2(1-p)$
estimates the information content of the system:

In [8]:
import math

entropy = analyzer.entropy_estimate()
avg_density = statistics.mean(densities)

print(f"=== Entropy Analysis ===")
print(f"  Average density p = {avg_density:.6f}")
print(f"  Binary entropy H(p) = {entropy:.4f} bits")
print(f"  Theoretical max (H(0.5)) = 1.0000 bits")
print(f"  Bernoulli map entropy (log₂2) = {math.log2(2):.4f} bits")
print(f"\n  Interpretation: {'high entropy (diverse)' if entropy > 0.8 else 'moderate entropy' if entropy > 0.5 else 'low entropy (sparse)'}")

=== Entropy Analysis ===
  Average density p = 0.000846
  Binary entropy H(p) = 0.0099 bits
  Theoretical max (H(0.5)) = 1.0000 bits
  Bernoulli map entropy (log₂2) = 1.0000 bits

  Interpretation: low entropy (sparse)


## 7. Mixing Rate — How Fast Does the System Change?

The mixing rate measures the average fraction of bits that change
per transition. High mixing = the system explores state space rapidly:

In [9]:
mixing = analyzer.mixing_rate()

print(f"=== Mixing Analysis ===")
print(f"  Mixing rate: {mixing:.6f} (fraction of bits changed per step)")
print(f"  Avg bits changed: {mixing * N_BITS:.1f} out of {N_BITS}")

# Per-transition costs
costs = [t.hamming_cost for t in analyzer.transitions]
print(f"\n  Per-transition Hamming costs:")
for i, c in enumerate(costs):
    bar = "█" * (c // 2)
    print(f"    Step {i:2d}→{i+1:2d}: {c:5d} bits  {bar}")

=== Mixing Analysis ===
  Mixing rate: 0.000177 (fraction of bits changed per step)
  Avg bits changed: 5.8 out of 32768

  Per-transition Hamming costs:
    Step  0→ 1:     5 bits  ██
    Step  1→ 2:     8 bits  ████
    Step  2→ 3:     5 bits  ██
    Step  3→ 4:     8 bits  ████
    Step  4→ 5:     6 bits  ███
    Step  5→ 6:     4 bits  ██
    Step  6→ 7:     4 bits  ██
    Step  7→ 8:     7 bits  ███
    Step  8→ 9:     3 bits  █
    Step  9→10:     9 bits  ████
    Step 10→11:     4 bits  ██
    Step 11→12:     6 bits  ███
    Step 12→13:     6 bits  ███
    Step 13→14:     6 bits  ███


## 8. Recurrence Time — Poincaré's Theorem

In ergodic systems, every state eventually recurs.
The `recurrence_time()` finds when the system first returns
"close" to the initial state:

In [10]:
# Try different thresholds
for threshold in [0.05, 0.10, 0.15, 0.20, 0.30]:
    t = analyzer.recurrence_time(threshold=threshold)
    status = f"step {t}" if t is not None else "not observed"
    print(f"  Recurrence (threshold={threshold:.2f}): {status}")

print(f"\nNote: With short trajectories ({len(analyzer.trajectory)} steps),")
print(f"recurrence may not be observed. Longer runs → better estimates.")

  Recurrence (threshold=0.05): step 1
  Recurrence (threshold=0.10): step 1
  Recurrence (threshold=0.15): step 1
  Recurrence (threshold=0.20): step 1
  Recurrence (threshold=0.30): step 1

Note: With short trajectories (15 steps),
recurrence may not be observed. Longer runs → better estimates.


## 9. Lyapunov Estimate — Sensitivity to Perturbation

The Lyapunov exponent $\lambda$ measures sensitivity to initial conditions:
- $\lambda > 0$: chaotic (nearby trajectories diverge exponentially)
- $\lambda = 0$: marginal (neutral stability)
- $\lambda < 0$: contracting (trajectories converge)

For the classical Bernoulli map, $\lambda = \ln 2 \approx 0.693$:

In [11]:
lyap = analyzer.lyapunov_estimate()

print(f"=== Lyapunov Analysis ===")
print(f"  Estimated Lyapunov exponent: {lyap:.4f}")
print(f"  Bernoulli map reference:     {math.log(2):.4f} (ln 2)")
print(f"  Ratio to Bernoulli:          {lyap / math.log(2):.2f}x" if math.log(2) != 0 else "")
print(f"\n  Interpretation: ", end="")
if lyap > 0.5:
    print("strongly chaotic (rapid divergence)")
elif lyap > 0:
    print("weakly chaotic (slow divergence)")
elif lyap == 0:
    print("neutral (marginal stability)")
else:
    print("contracting (trajectories converge)")

=== Lyapunov Analysis ===
  Estimated Lyapunov exponent: 1.7104
  Bernoulli map reference:     0.6931 (ln 2)
  Ratio to Bernoulli:          2.47x

  Interpretation: strongly chaotic (rapid divergence)


## 10. Walk Complexity — Full Structural Summary

In [12]:
wc = analyzer.walk_complexity()

print("=== De Bruijn Walk Complexity ===")
for k, v in wc.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    else:
        print(f"  {k:25s}: {v}")

=== De Bruijn Walk Complexity ===
  length                   : 15
  unique_signatures        : 12
  avg_transition_cost      : 5.7857
  max_transition_cost      : 9
  min_transition_cost      : 3
  total_bits_changed       : 81
  entropy_estimate         : 0.0099
  mixing_rate              : 0.0002


In [ ]:
# Human-readable summary
print(analyzer.summary())

## 11. UnifiedSystemController — Bridging Both Worlds

The `UnifiedSystemController` accepts both HLLSet and BitVector inputs,
combining the algebra layer with the symbolic dynamics layer:

In [13]:
# Create unified controller
uc = UnifiedSystemController(n_bits=N_BITS, mode=SteeringMode.INCREMENTAL)

# Set target
target = HLLSet.from_batch(["goal", "state", "tokens"], p_bits=P_BITS)
uc.set_target_hllset(target)

# Observe a sequence of HLLSets
states = [
    HLLSet.from_batch(["far", "from", "goal"], p_bits=P_BITS),
    HLLSet.from_batch(["closer", "to", "goal", "state"], p_bits=P_BITS),
    HLLSet.from_batch(["goal", "state", "tokens"], p_bits=P_BITS),  # = target
]

print(f"{'Step':>4}  {'Density':>8}  {'Pop':>6}  {'Cost':>5}  {'Dist':>8}  At Target")
print("-" * 55)

for i, hll in enumerate(states):
    result = uc.observe_hllset(hll)
    cost = result.get("last_transition_cost", "—")
    dist = result.get("distance_to_target", "—")
    at_target = result.get("at_target", "—")
    
    cost_str = f"{cost:5d}" if isinstance(cost, int) else f"{'—':>5}"
    dist_str = f"{dist:.4f}" if isinstance(dist, float) else f"{'—':>8}"
    
    print(f"{i:4d}  {result['state_density']:.6f}  {result['state_popcount']:6d}  "
          f"{cost_str}  {dist_str}  {at_target}")

Step   Density     Pop   Cost      Dist  At Target
-------------------------------------------------------
   0  0.000092       3      —  0.0001  True
   1  0.000122       4      5  0.0001  True
   2  0.000092       3      3  0.0000  True


In [14]:
# Get the embedded analyzer's summary
print(uc.analysis_summary())

=== Bernoulli / Symbolic Dynamics Analysis ===
Trajectory length: 3
Unique signatures: 2
Entropy estimate: 0.001 bits
Mixing rate: 0.000
Lyapunov estimate: 1.354
Recurrence time: 1
Total bits changed: 8


## 12. Computing Shift Transitions

The `UnifiedSystemController` can compute the shift needed to reach the target:

In [15]:
# Compute shift from current state to target
current_bv = BitVectorState.from_hllset(
    HLLSet.from_batch(["not", "at", "target"], p_bits=P_BITS)
)
target_bv = BitVectorState.from_hllset(target)

shift = ShiftTransition.from_states(current_bv, target_bv)
print(f"Shift to target:")
print(f"  Bits to clear: {shift.clear_mask.popcount()}")
print(f"  Bits to set:   {shift.set_mask.popcount()}")
print(f"  Hamming cost:  {shift.hamming_cost}")
print(f"  Is identity:   {shift.is_identity}")

# Apply and verify
result = shift.apply(current_bv)
print(f"\nAfter applying shift:")
print(f"  Distance to target: {result.hamming_distance(target_bv)}")

Shift to target:
  Bits to clear: 3
  Bits to set:   3
  Hamming cost:  6
  Is identity:   False

After applying shift:
  Distance to target: 0


## 13. Comparing Trajectories — Divergence Analysis

Compare two trajectories that start from the same state
but evolve differently (sensitivity to perturbation):

In [16]:
random.seed(123)

# Two analyzers starting from same state
a1 = BernoulliAnalyzer(n_bits=N_BITS)
a2 = BernoulliAnalyzer(n_bits=N_BITS)

# Same initial state
initial = HLLSet.from_batch([f"seed_{i}" for i in range(15)], p_bits=P_BITS)
a1.observe_hllset(initial)
a2.observe_hllset(initial)

# Evolve with slightly different random streams
tokens1 = set(f"seed_{i}" for i in range(15))
tokens2 = set(f"seed_{i}" for i in range(15))

for step in range(12):
    # Trajectory 1: random evolution
    random.seed(1000 + step)
    add1 = {f"t1_{step}_{j}" for j in range(random.randint(2, 4))}
    del1 = set(random.sample(list(tokens1), min(2, len(tokens1))))
    tokens1 = (tokens1 - del1) | add1
    a1.observe_hllset(HLLSet.from_batch(list(tokens1), p_bits=P_BITS))
    
    # Trajectory 2: slightly perturbed
    random.seed(2000 + step)
    add2 = {f"t2_{step}_{j}" for j in range(random.randint(2, 4))}
    del2 = set(random.sample(list(tokens2), min(2, len(tokens2))))
    tokens2 = (tokens2 - del2) | add2
    a2.observe_hllset(HLLSet.from_batch(list(tokens2), p_bits=P_BITS))

# Compare divergence
print(f"{'Step':>4}  {'Dist(s1,s2)':>12}  {'d1':>8}  {'d2':>8}")
print("-" * 40)
for i in range(len(a1.trajectory)):
    dist = a1.trajectory[i].normalized_distance(a2.trajectory[i])
    d1 = a1.trajectory[i].density
    d2 = a2.trajectory[i].density
    print(f"{i:4d}  {dist:12.6f}  {d1:8.6f}  {d2:8.6f}")

print(f"\nTrajectory 1: entropy={a1.entropy_estimate():.4f}, mixing={a1.mixing_rate():.6f}")
print(f"Trajectory 2: entropy={a2.entropy_estimate():.4f}, mixing={a2.mixing_rate():.6f}")

Step   Dist(s1,s2)        d1        d2
----------------------------------------
   0      0.000000  0.000458  0.000458
   1      0.000244  0.000488  0.000488
   2      0.000549  0.000488  0.000549
   3      0.000732  0.000549  0.000610
   4      0.000885  0.000580  0.000610
   5      0.000977  0.000610  0.000671
   6      0.001160  0.000641  0.000702
   7      0.001251  0.000671  0.000763
   8      0.001251  0.000671  0.000763
   9      0.001343  0.000732  0.000732
  10      0.001434  0.000732  0.000763
  11      0.001526  0.000793  0.000793
  12      0.001617  0.000824  0.000854

Trajectory 1: entropy=0.0076, mixing=0.000153
Trajectory 2: entropy=0.0081, mixing=0.000150


## 14. The Full Picture — From Bits to Meaning

This tutorial completes the journey from concrete data structures
to abstract dynamical theory:

```
Tier 1: Foundation                  Tier 2: Coordination
┌───────────────────┐               ┌───────────────────┐
│ HLLSet, BitVector │──registers──▶│ Registry, Store,  │
│ BSS(τ, ρ)         │               │ Ring Transactions │
└───────────────────┘               └───────────────────┘
         │                                  │
         ▼                                  ▼
Tier 3: Disambiguation              Tier 4: System Dynamics
┌───────────────────┐               ┌───────────────────┐
│ De Bruijn, D/R/N  │──evolution──▶│ Noether (observe) │
│ Disambiguation    │   phases      │ Plan/Steer (act)  │
│ Lattice Reconstr. │               │ Bernoulli (theory)│
└───────────────────┘               └───────────────────┘
```

**The unifying equation:**

$$R(t+1) = [R(t) \setminus D(t)] \cup N(t)$$

appears at every level:
- **Tier 1:** Set operations on HLLSets
- **Tier 3:** D/R/N decomposition of transitions
- **Tier 4 (Observe):** Noether conservation: $\Phi(t) = |N| - |D|$
- **Tier 4 (Act):** Steering: minimize cost $|D| + |N|$
- **Tier 4 (Theory):** Bernoulli shift: clear/set masks on $\{0,1\}^N$

---

## Summary & Key Takeaways

### BitVectorState — Points in Binary Space

| API | Purpose |
|-----|---------|
| `BitVectorState.from_hllset(h)` | Convert HLLSet to bit vector |
| `.popcount`, `.density` | Hamming weight, fraction of 1-bits |
| `.xor()`, `.and_()`, `.or_()` | Ring operations |
| `.hamming_distance()` | Number of differing bits |
| `.normalized_distance()` | Hamming distance / n_bits |

### ShiftTransition — Generalized Shift

| API | Purpose |
|-----|---------|
| `ShiftTransition.from_states(s1, s2)` | Compute transition |
| `ShiftTransition.from_drn(drn)` | Convert from D/R/N triple |
| `.apply(state)` | Apply shift to get new state |
| `.hamming_cost` | Total bits changed |
| `.is_identity` | No bits changed? |

### BernoulliAnalyzer — Ergodic Theory

| API | Purpose |
|-----|---------|
| `.observe_hllset(h)` | Add state to trajectory |
| `.density_series()` | Bit density time series |
| `.entropy_estimate()` | Binary entropy $H(p)$ |
| `.mixing_rate()` | Fraction of bits changed per step |
| `.recurrence_time(θ)` | Poincaré recurrence |
| `.lyapunov_estimate()` | Sensitivity to perturbation |
| `.walk_complexity()` | Full structural stats |
| `.summary()` | Human-readable report |

### UnifiedSystemController

| API | Purpose |
|-----|---------|
| `.set_target_hllset(h)` | Set target in either representation |
| `.observe_hllset(h)` | Observe and analyze |
| `.compute_shift(current)` | Bit-level shift to target |
| `.analysis_summary()` | Full Bernoulli analysis |

### Key Theoretical Connections

| HLLSet Concept | Symbolic Dynamics Analog |
|----------------|--------------------------|
| D/R/N triple | Clear/Retain/Set on $\{0,1\}^N$ |
| BSS τ | Overlap measure in Hamming space |
| Noether flux Φ | Net bit change rate |
| Evolution phase | Ergodic phase classification |
| Lattice level | Orbit complexity class |

---

**🎓 Congratulations!** You've completed the full HLLSet Algebra course.

From Tutorial 01's first `HLLSet.from_batch()` to Tutorial 15's
Lyapunov exponents, you've seen how a single probabilistic data
structure generates a complete algebraic and dynamical framework.

**The course:**
- **Tier 1 (01–04):** Foundation — HLLSet, BSS, Tensor, Lattice, Store
- **Tier 2 (05–07):** Coordination — Registry, Transactions, BSS Metrics
- **Tier 3 (08–12):** Disambiguation — De Bruijn, Disambiguation, Lattice Reconstruction
- **Tier 4 (13–15):** System Dynamics — Observe, Act, Theory